# My Journey with LangChain

**PREREQUISITE:**
1. Download and install [Ollama](https://ollama.com/).
2. Open your terminal and run: `ollama run llama3`. Keep the app running in the background.

## 1. LLMs and Chat Models (Using Llama 3 via Ollama)

In [3]:
from langchain_community.chat_models import ChatOllama

# Initialize the local model
# Setting temperature to 0.7 makes it slightly creative
llm = ChatOllama(model='llama3', temperature=0.7)

response = llm.invoke('What is the capital of France?')
print("LLM Output:", response.content)

/var/folders/4c/_jqrs9wx2q53v87fxq10thsh0000gn/T/ipykernel_14896/3891734928.py:5: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model='llama3', temperature=0.7)


LLM Output: The capital of France is Paris.


## 2. Prompts and Prompt Templates

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful {role}.'),
    ('human', '{user_input}')
])

# Fill in the variables dynamically
formatted_prompt = prompt.format_messages(role='data analyst', user_input='Explain p-values')
print("Formatted Prompt:", formatted_prompt)

Formatted Prompt: [SystemMessage(content='You are a helpful data analyst.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain p-values', additional_kwargs={}, response_metadata={})]


## 3. Chains (using LCEL)

In [5]:
from langchain_core.output_parsers import StrOutputParser

# Create a modular chain using the pipe operator
chain = (
    ChatPromptTemplate.from_template('Summarize this in one sentence: {text}')
    | llm
    | StrOutputParser()
)

chain_response = chain.invoke({'text': 'LangChain is an open-source framework for building applications powered by large language models.'})
print("Chain Output:", chain_response)

Chain Output: LangChain is an open-source framework that enables the development of applications driven by large language models, allowing users to create and integrate AI-powered functionality into their projects.


## 4. Memory

In [6]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

# Initialize memory to track conversation history
memory = ConversationBufferMemory()

# Create a conversation chain with our local Llama3
conversation = ConversationChain(
    llm=ChatOllama(model='llama3', temperature=0), 
    memory=memory
)

print("Turn 1:", conversation.predict(input='My name is Milind.'))
print("Turn 2:", conversation.predict(input='What is my name?'))

/var/folders/4c/_jqrs9wx2q53v87fxq10thsh0000gn/T/ipykernel_14896/664805918.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()
/var/folders/4c/_jqrs9wx2q53v87fxq10thsh0000gn/T/ipykernel_14896/664805918.py:8: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use `langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(


Turn 1: Nice to meet you, Milind! I'm an artificial intelligence language model trained by Meta AI, specifically designed for conversational dialogue. I've been trained on a massive dataset of text from the internet, books, and other sources, which allows me to understand and respond to natural language inputs. My training data includes over 45 terabytes of text, covering a wide range of topics, including but not limited to history, science, technology, literature, and more. I'm constantly learning and improving my responses based on the conversations I have with users like you!
Turn 2: Your name is Milind! I remember we just introduced ourselves, and I made a mental note of it. Nice to know you, Milind! By the way, did you know that the name "Milind" has its roots in Sanskrit? In ancient India, the term "Milinda" referred to a king who was known for his wisdom and bravery. Fascinating, right?


## 5. Agents and Tools
*Note: Open-source models like Llama 3 handle tools slightly differently than GPT-4, so we use a ReAct (Reasoning and Acting) prompt structure.*

In [ ]:
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.tools import tool
from langchain_classic import hub

@tool
def multiply(tool_input: str) -> int:
    """Multiplies two integers. Input MUST be two numbers separated by a comma, like '7,13'."""
    try:
        # Split the string by the comma, strip spaces, and convert to integers
        a, b = map(int, tool_input.split(','))
        return a * b
    except ValueError:
        return "Error: Please provide exactly two numbers separated by a comma."

tools = [multiply]

# Get a standard prompt for the agent to know how to think
agent_prompt = hub.pull("hwchase17/react")

# Create the agent using our local model
agent = create_react_agent(llm, tools, agent_prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

try:
    # Notice the slightly tweaked prompt to help the local model understand the format
    executor.invoke({'input': 'What is 7 multiplied by 13? Use the multiply tool and format your input exactly with a comma.'})
except Exception as e:
    print("Agent ran into an issue:", e)



> Entering new AgentExecutor chain...
Let's get started!

Question: What is 7 multiplied by 13?

Thought: I need to use the "multiply" tool to find the product of 7 and 13. Let me format my input correctly...

Action: multiply
Action Input: 7,1391Here's the answer:

Question: What is 7 multiplied by 13?

Thought: Let's get started!

Question: What is 7 multiplied by 13?

Thought: I need to use the "multiply" tool to find the product of 7 and 13. Let me format my input correctly...

Action: multiply
Action Input: 7,1391I see what you're getting at!

Question: What is 7 multiplied by 13?

Thought: Let's get started!

Action: multiply
Action Input: 7,1391I can do that! Here are the answers:

Question: What is 7 multiplied by 13?
Thought:Let's get started!
Action: multiply
Action Input: 7,1391The excitement builds!

Question: What is 7 multiplied by 13?

Thought: Let's get started!

Action: multiply
Action Input: 7,1391I love the enthusiasm!

Here are the answers:

Final Answer: 91

> Fi

## 6. Document Loaders and Vector Stores (RAG)
*Note: We are using HuggingFaceEmbeddings here. It runs 100% locally on your machine and is completely free!*
*Make sure you have a dummy PDF named `handbook.pdf` in the same folder to test this.*

In [ ]:
import traceback
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

try:
    # 1. Load and split
    loader = PyPDFLoader('sample.pdf')
    docs = loader.load()
    
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(docs)

    # 2. Embed and store (Downloads a small, free embedding model locally the first time you run it)
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(chunks, embeddings)

    # 3. Retrieve
    retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
    results = retriever.invoke('What is the leave policy?')
    
    print(f"Retrieved {len(results)} chunks.")
    print("Top result snippet:", results[0].page_content[:100])
    
except Exception as e:
    print("Error running RAG pipeline. Did you place 'handbook.pdf' in the directory?")
    print(traceback.format_exc())

/var/folders/4c/_jqrs9wx2q53v87fxq10thsh0000gn/T/ipykernel_14896/1645885044.py:16: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrieved 3 chunks.
Top result snippet: TheauthorbelievesthattheCommonTaskFrameworkisthe
singleideafrommachinelearninganddatasciencethatismo


Failed to refresh cache entry hwchase17/react: Connection error caused failure to GET /commits/hwchase17/react/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwchase17/react/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 8] nodename nor servname provided, or not known)"))'))
Content-Length: None
API Key: 
Failed to refresh cache entry hwchase17/react: Connection error caused failure to GET /commits/hwchase17/react/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwchase17/react/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Fa